# Ilerda: entrenar la NNUE amb les dades de la 0.15.3 mes les velles amb taules de finals (768 -> HIDDEN -> 1, 8 buckets de sortida, bullet)
Mateixa xarxa que `ilerda-train-buckets.ipynb`, amb les dades noves de `tools/prepara` (`nou-00.txt` ... `nou-05.txt`: sense repetides i **ja barrejades**, uns 88 M de posicions per tros). Com que ja venen barrejades, aqui nomes es converteixen; no cal `shuffle`, que no cabria al disc.

Accelerator = GPU T4, Internet = On. Afegeix el dataset amb els trossos; la cel·la 2 els troba sola. `TROSSOS` diu quants se'n fan servir (6 = tots). Les xarxes surten a `/kaggle/working/ilerda-<HIDDEN>-b8-sb<N>.bin`.

In [ ]:
!nvidia-smi
!df -h /tmp /kaggle/working
!curl --proto =https --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y 2>&1 | tail -3

In [ ]:
import glob
HIDDEN = 256       # 256 o 512
TROSSOS = 6        # quants trossos de dades (1 a 6)
SUPERBATCHES = 48  # 1 superbatch = 100 M de posicions; amb 518 M de noves + 109 M de velles, 48 son unes 7,6 passades
PARTS = sorted(glob.glob('/kaggle/input/**/nou-0*.txt', recursive=True))[:TROSSOS]
if not PARTS:  # si Kaggle no ha descomprimit els zips, es descomprimeixen d'un en un a la cel·la de conversio
    PARTS = sorted(glob.glob('/kaggle/input/**/nou-0*.zip', recursive=True))[:TROSSOS]
print(len(PARTS), 'trossos'); print('\n'.join(PARTS))
assert len(PARTS) == TROSSOS, 'no trobo els trossos: afegeix el dataset al notebook'
FINALS = sorted(glob.glob('/kaggle/input/**/finals-0*.txt', recursive=True))
print('finals (posicions de 7-12 peces jugades amb taules):', FINALS)
assert len(FINALS) == 1, 'no trobo finals-00.txt: afegeix el dataset srobis/ilerda-nou-tb'
USAR_FINALS = False  # False: nomes les dades noves amb taules i les velles, per aillar l'efecte dels finals
if USAR_FINALS: PARTS += FINALS
VELLES = glob.glob('/kaggle/input/**/mix2-tb.txt', recursive=True)
print('dades velles (amb correccio de taules de finals):', VELLES)
assert len(VELLES) == 1, 'no trobo mix2-tb.txt: afegeix el dataset srobis/ilerda-mix2-tb'

In [ ]:
import os
os.environ['PATH'] += ':/root/.cargo/bin'
!git clone --depth 1 https://github.com/jw1912/bullet /kaggle/working/bullet 2>&1 | tail -2
!cd /kaggle/working/bullet && git log -1 --format='bullet %h %cd' && cargo b -r --package bullet-utils 2>&1 | tail -3

In [ ]:
# Converteix cada tros al format binari de bullet (32 bytes per posicio) a /tmp, que no compta per al limit de 20 GB de /kaggle/working.
import subprocess
os.makedirs('/tmp/dades', exist_ok=True)
BINS = []
for p in PARTS:
    out = '/tmp/dades/' + os.path.basename(p)[:6] + '.bullet'
    zipped = p.endswith('.zip')
    if zipped:
        subprocess.run(['unzip', '-o', '-q', p, '-d', '/tmp/txt'], check=True)
        p = '/tmp/txt/' + os.path.basename(p)[:6] + '.txt'
    r = subprocess.run(['/kaggle/working/bullet/target/release/bullet-utils', 'convert', '--from', 'text', '--input', p, '--output', out, '--threads', '4'], capture_output=True, text=True)
    assert os.path.exists(out), (r.stdout + r.stderr)[-3000:]
    print(os.path.basename(p), '->', os.path.getsize(out) // 32, 'posicions', (r.stdout + r.stderr).strip().splitlines()[-1:])
    BINS.append(out)
    if zipped: os.remove(p)
# Les dades velles (0.15.0, amb la puntuacio corregida amb taules de finals) no venen barrejades: es converteixen i es barregen.
U = '/kaggle/working/bullet/target/release/bullet-utils'
r = subprocess.run([U, 'convert', '--from', 'text', '--input', VELLES[0], '--output', '/tmp/dades/velles0.bullet', '--threads', '4'], capture_output=True, text=True)
assert os.path.exists('/tmp/dades/velles0.bullet'), (r.stdout + r.stderr)[-3000:]
r = subprocess.run([U, 'shuffle', '--input', '/tmp/dades/velles0.bullet', '--mem-used-mb', '8000', '--output', '/tmp/dades/velles.bullet'], capture_output=True, text=True)
assert os.path.exists('/tmp/dades/velles.bullet'), (r.stdout + r.stderr)[-3000:]
os.remove('/tmp/dades/velles0.bullet')
print('velles ->', os.path.getsize('/tmp/dades/velles.bullet') // 32, 'posicions')
# Tot junt en un sol fitxer, agafant a l'atzar de cada un (interleave), perque les velles quedin repartides entre les noves.
print(subprocess.run([U, 'interleave', '--help'], capture_output=True, text=True).stdout[-1500:])
TOT = '/tmp/dades/tot.bullet'
r = subprocess.run([U, 'interleave'] + BINS + ['/tmp/dades/velles.bullet', '--output', TOT], capture_output=True, text=True)
print((r.stdout + r.stderr)[-1500:])
assert os.path.exists(TOT) and os.path.getsize(TOT) == sum(os.path.getsize(b) for b in BINS) + os.path.getsize('/tmp/dades/velles.bullet'), 'interleave ha fallat'
for b in BINS + ['/tmp/dades/velles.bullet']: os.remove(b)
print('total ->', os.path.getsize(TOT) // 32, 'posicions')
BINS = [TOT]
!df -h /tmp

In [ ]:
from pathlib import Path
RS = r'''// Exemple de bullet per a Ilerda: (768 -> HIDDEN_SIZE)x2 -> 1 amb 8 buckets de sortida per nombre de peces.
// Mateixa arquitectura i mateixa forma de calendari que la xarxa de la 0.15.0 (StepLR 0,001 x0,1, WDL 0,75);
// canvien les dades (0.15.3, 5.000 nodes) i la durada, que es proporcional a la quantitat de dades.
use bullet_lib::{
    game::{inputs::Chess768, outputs::MaterialCount},
    nn::optimiser::AdamW,
    trainer::{
        save::SavedFormat,
        schedule::{TrainingSchedule, TrainingSteps, lr, wdl},
        settings::LocalSettings,
    },
    value::{ValueTrainerBuilder, loader::DirectSequentialDataLoader},
};

const HIDDEN_SIZE: usize = __HIDDEN__;
const NUM_OUTPUT_BUCKETS: usize = 8;
const SCALE: i32 = 400;
const QA: i16 = 255;
const QB: i16 = 64;

fn main() {
    let mut trainer = ValueTrainerBuilder::default()
        .dual_perspective()
        .optimiser(AdamW)
        .inputs(Chess768)
        .output_buckets(MaterialCount::<NUM_OUTPUT_BUCKETS>)
        .save_format(&[
            SavedFormat::id("l0w").round().quantise::<i16>(QA),
            SavedFormat::id("l0b").round().quantise::<i16>(QA),
            // .transpose(): pesos de sortida [bucket][2N], contigus per bucket (el que llegeix Nnue.cs)
            SavedFormat::id("l1w").round().quantise::<i16>(QB).transpose(),
            SavedFormat::id("l1b").round().quantise::<i16>(QA * QB),
        ])
        .loss_fn(|output, target| output.sigmoid().squared_error(target))
        .build(|builder, stm_inputs, ntm_inputs, output_buckets| {
            let l0 = builder.new_affine("l0", 768, HIDDEN_SIZE);
            let l1 = builder.new_affine("l1", 2 * HIDDEN_SIZE, NUM_OUTPUT_BUCKETS);
            let stm_hidden = l0.forward(stm_inputs).screlu();
            let ntm_hidden = l0.forward(ntm_inputs).screlu();
            l1.forward(stm_hidden.concat(ntm_hidden)).select(output_buckets)
        });

    let schedule = TrainingSchedule {
        net_id: "ilerda".to_string(),
        eval_scale: SCALE as f32,
        steps: TrainingSteps { batch_size: 16_384, batches_per_superbatch: 6104, start_superbatch: 1, end_superbatch: __SB__ },
        wdl_scheduler: wdl::ConstantWDL { value: 0.75 },
        lr_scheduler: lr::StepLR { start: 0.001, gamma: 0.1, step: __STEP__ },
        save_rate: __SAVE__,
    };

    let settings = LocalSettings { threads: 4, test_set: None, output_directory: "/kaggle/working/checkpoints", batch_queue_size: 64 };
    let data_loader = DirectSequentialDataLoader::new(&[__FILES__]);

    trainer.run(&schedule, &settings, &data_loader);
}
'''
RS = RS.replace('__HIDDEN__', str(HIDDEN)).replace('__SB__', str(SUPERBATCHES))
RS = RS.replace('__STEP__', str(max(1, round(SUPERBATCHES * 7 / 20)))).replace('__SAVE__', str(max(1, SUPERBATCHES // 4)))
RS = RS.replace('__FILES__', ', '.join('"' + b + '"' for b in BINS))
Path('/kaggle/working/bullet/examples/progression/2_output_buckets.rs').write_text(RS)

In [ ]:
!cd /kaggle/working/bullet && export PATH=$PATH:/root/.cargo/bin CUDA_PATH=$(ls -d /usr/local/cuda-* | sort -V | tail -1) RUSTFLAGS='-L /usr/local/nvidia/lib64' && echo $CUDA_PATH && cargo r -r --example 2_output_buckets --features cuda 2>&1 | tail -n 80

In [ ]:
# Copia la xarxa final i les intermedies (per poder comparar quanta durada cal).
import shutil
for d in sorted(glob.glob('/kaggle/working/checkpoints/ilerda-*')):
    sb = d.rsplit('-', 1)[1]
    out = f'/kaggle/working/ilerda-{HIDDEN}-b8-tbnf-sb{sb}.bin'
    shutil.copy(d + '/quantised.bin', out)
    print(out, os.path.getsize(out))
# Neteja: a la sortida del notebook nomes hi han de quedar les xarxes (el clon de bullet i la compilacio pesen uns quants GB).
shutil.rmtree('/kaggle/working/bullet', ignore_errors=True)
shutil.rmtree('/kaggle/working/checkpoints', ignore_errors=True)